In [9]:
import os
import regex
import json
import time, datetime, pytz
#import speech_recognition as sr

from langchain_mistralai import ChatMistralAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage, AIMessage
from langchain.tools import tool, StructuredTool

from db_search import get_info
import knowledge_base as RAGer
#from text_to_speech import Text2Speech

In [10]:
import spacy
from sentence_transformers import SentenceTransformer, util

B:\Software\Anaconda\envs\agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2,
    api_key="HD8VnqYHeT0V9TYnbixOFmv59cTBtc5l"
)

In [12]:
speaker = Text2Speech()

NameError: name 'Text2Speech' is not defined

In [13]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 8.4 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 10.4 MB/s eta 0:00:01
     ------------------ --------------------- 5.8/12.8 MB 10.4 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 10.3 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 10.9 MB/s eta 0:00:01
     ------------------------------------ --- 11.5/12.8 MB 9.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 9.8 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.15.2
    Uninstalling typer-0.15.2:
      Successfully uninstalled typer-0.15.2
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [14]:
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

B:\Software\Anaconda\envs\agent\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shrey\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [15]:
intent_examples = {
    "greet": ["Hello", "Hi", "Hey", "Good morning", "Good evening", "Hi there", "Hello, is anyone there?", "Hey, can you help me?"],
    "goodbye": ["Bye", "Goodbye", "See you later", "Talk to you soon", "Have a nice day", "See you", "Thanks", "Thank you so much"],
    "loan_details": ["Can you tell me my loan details?", "What is my outstanding loan balance?", "How much loan amount is left to pay?", "I want to know my loan status", "Provide my loan details", "How much do I owe?"],
    "payment_due": ["When is my next payment due?", "Tell me my EMI due date", "What is my next EMI date?", "I need my loan payment schedule", "Can you remind me of my payment date?"],
    "make_payment": ["I want to pay my EMI", "How can I pay my loan?", "Send me the payment link", "Can I pay my EMI online?", "Where can I make the payment?"],
    "grace_period": ["Can I get an extension on my loan payment?", "I need a grace period", "Can I delay my EMI?", "I need some time to make my payment", "Is there a penalty for late payment?"],
    "financial_difficulty": ["I am facing financial issues", "I am unable to pay the loan right now", "My salary is delayed, I need help", "Can I restructure my loan?", "I need more time to make the payment"],
    "interest_rate": ["What is my loan interest rate?", "Can you tell me the interest rate on my loan?", "Has my interest rate changed?", "How is my interest calculated?", "I need details on my loan interest"],
    "late_fees": ["What are the penalties for late payment?", "Will I be charged a fee if I miss my EMI?", "How much is the late fee?", "What happens if I don’t pay on time?", "Explain the overdue charges"],
    "loan_tenure": ["What is the duration of my loan?", "How many EMIs are left?", "How long do I have to repay the loan?", "When will my loan be fully paid?", "Tell me about my repayment period"],
    "dispute_transaction": ["I see an incorrect charge on my loan statement", "My loan balance is wrong", "I was charged extra, please check", "I have a dispute regarding my loan", "Why was I charged more than usual?"],
    "confirm_payment": ["Did you receive my last EMI payment?", "Has my loan payment been processed?", "I made the payment, please confirm", "My money has been deducted but loan not updated", "Can you check my last payment status?"],
    "loan_preclosure": ["Can I pay off my loan early?", "What is the preclosure process?", "How much do I need to pay to close my loan?", "Can I settle my loan early?", "What are the charges for early loan repayment?"],
    "contact_support": ["I need to speak with a customer representative", "Can I talk to an agent?", "Provide customer support details", "I need help from a loan officer", "Who can assist me with my loan issue?"],
    "assistance": ["I need help with my loan", "Can you assist me?", "I need guidance on loan repayment", "Can someone help me with my account?", "Help me with my financial queries"],
    "enquiry": ["I have a question about my loan", "Can you provide more details?", "I need clarification on loan policies", "Tell me about the repayment process", "What are the terms of my loan?"],
    "fallback": ["I don’t understand", "This doesn’t make sense", "Can you repeat that?", "I need more information", "Sorry, I didn’t get that"]
}

In [16]:
intent_embeddings = {key: model.encode(sentences, convert_to_tensor=True) for key, sentences in intent_examples.items()}

In [17]:
#Refresh convo.

template = """You are a professional credit card payment management executive.
Your primary responsibility is to assist customers with understanding their loan details, 
provide helpful reminders about upcoming payments, and ensure a smooth repayment experience. 

You are helping our customer {first_name} {last_name} to remind them of their outstanding balance along with minimum due amount. 
The goal is to obtain promise to pay date, and amount from willing customers, persuade unwilling customers to make payment. 
You may provide EMI offer to eligible customers. Communication needs to be adjusted based on number of days for due date. 
Stay focused on this context and provide relevant information. 
Do not invent information not drawn from the context. Answer only questions related to the context.

Rules of communication:
1. Maintain polite, non-confrontational tone
2. Handle concerns empathetically
3. Use clear and respectful language
4. Keep responses short and natural
5. Keep responses very short and to the point, mimicking human-like conversation.  
6. Prioritize user well-being and information accuracy.  
7. Show understanding with acknowledgments.  
8. Avoid long statements, especially while ending the call.  
9. Keep the conversation short and avoid unnecessary remarks.  
10. Avoid repeating borrower’s answers.  
11. Avoid repetitive phrases and statements.
12. Do not repeat a sentence twice.  
13. Avoid speculative/unverified information
14. Only mention numbers/amounts specified in relevant data.
15. Do not confront customers about anything and keep your tone polite.  
16. Maintain unwavering professionalism.  
17. Do not disclose any details to anyone except the customer.  
18. Protect user privacy and safety.  
19. Provide helpful, ethical, and constructive assistance.  
20. Recognize and gracefully handle inappropriate requests.  

Policies to adhere to: {policies}
"""

chat_history = []

In [18]:
template_messages = [
    SystemMessage(content=template),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{query}")
    #HumanMessage(content="{query}")
]

prompt_template = ChatPromptTemplate.from_messages(template_messages)

In [19]:
@tool

def get_user_data() -> dict:
    """
    Returns all information about the customer's loan.
    """
    key = int(phone)
    user_data = get_info(key)
    return user_data

In [20]:
@tool
def current_date_time() -> dict:
    '''
    Returns the current server date and time in JSON format.
    '''
    now = datetime.datetime.now()
    ist_timezone = pytz.timezone('Asia/Kolkata')
    dt_ist = now.astimezone(ist_timezone)
    time = dict()
    time['day'] = dt_ist.strftime('%A')
    time['month'] = dt_ist.strftime('%B')
    time['date'] = dt_ist.strftime('%Y-%m-%d')
    time['time'] = dt_ist.strftime('%H:%M')
    return time

In [21]:
llm_with_tools = llm.bind_tools([get_user_data, current_date_time])

tool_mapping = {
    "get_user_data" : get_user_data,
    "current_date_time" : current_date_time
}

In [22]:
def analyze_text(text):
    doc = nlp(text)

    # Extract relevant entities
    entities = {ent.label_: ent.text for ent in doc.ents if ent.label_ in ["ORG", "GPE", "PERSON", "PRODUCT", "MONEY", "DATE", "CARDINAL", "EVENT", "TIME"]}

    # Encode input text for intent matching
    input_embedding = model.encode(text, convert_to_tensor=True)

    # Find best matching intent
    best_intent = "unknown"
    best_score = 0.4  # Confidence threshold

    for intent, embeddings in intent_embeddings.items():
        similarity_score = util.pytorch_cos_sim(input_embedding, embeddings).max().item()
        if similarity_score > best_score:
            best_intent = intent
            best_score = similarity_score

    return {"entities": entities, "intent": best_intent}

In [23]:
phone = 9804604602

user_data = get_info(phone)

user_info = {
    "first_name": user_data['first_name'],
    "last_name": user_data['last_name']
}

In [24]:
policies = RAGer.fetch_query('All policies related to loan')

B:\Software\Anaconda\envs\agent\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shrey\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [25]:
formatted_template = template.format(**user_info,policies=policies)
template_messages[0] = SystemMessage(content=formatted_template)
prompt_template = ChatPromptTemplate.from_messages(template_messages)

In [27]:
def chat(text:str) -> str:
    messages = prompt_template.format_messages(
        chat_history=chat_history,
        query = text
    )
    chat_history.append(HumanMessage(text))
    
    response = llm_with_tools.invoke(messages)
    chat_history.append(response)
    
    if response.tool_calls:
        for tool_call in response.tool_calls:
            tool = tool_mapping[tool_call['name']]  
            output = tool.invoke(tool_call['args'])
            
            chat_history.append(ToolMessage(str(output), tool_call_id=tool_call['id']))
        ai_says = llm_with_tools.invoke(chat_history)
        chat_history.append(ai_says)
    else:
        ai_says = response

    return ai_says.content

In [17]:
def recognize_speech():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("_"*100)
        try:
            audio = recognizer.listen(source, timeout=3000)
            user_input = recognizer.recognize_google(audio)
            print(f"You said: {user_input}")
            return user_input
        except sr.UnknownValueError:
            return None
        except sr.WaitTimeoutError:
            return None

In [28]:
welcome = AIMessage(content=f"Hello {user_info['first_name']}, I wanted to talk about your loan repayment.")
chat_history.append(welcome)
print('Bot: ',welcome.content)
#speaker.speak(welcome.content)

while True:
    #print('Listening')
    #query = recognize_speech()
    query = input('You: ')
    if query is not None:
        if query.lower() in ['okay thank you','ok thank you','ok thank you']:
            res = chat(query)
            print(res)
            #speaker.speak(res)
            break
        else:
            print(analyze_text(query))
            res = chat(query)
            cleaned_text = regex.sub(r'[\/@|!?]', '', res)
            print('Bot: ',cleaned_text)
            #speaker.speak(cleaned_text)
            

Bot:  Hello Diya, I wanted to talk about your loan repayment.


You:  Hello


{'entities': {}, 'intent': 'greet'}
Bot:  How are you doing today


You:  What is the time right now?


{'entities': {}, 'intent': 'grace_period'}
Bot:  The current date and time are March 6, 2025, at 22:19 on Thursday.


You:  Okay cool and how much loan do I have pending?


{'entities': {}, 'intent': 'interest_rate'}
Bot:  You have a pending loan amount of ₹25,866.40. Here are the details:

• Loan Type: Consumer Durable Loan
• Original Loan Amount: ₹36,100.60
• Interest Rate: 12.40%
• Installment Amount: ₹6,236.20
• Tenure: 6 months
• Start Date: 2024-05-09
• Balance to Pay: ₹25,866.40
• Payment Mode: Debit Card
• Last Payment Date: 2024-08-03
• Late Payments: None


You:  ok thank you


You're welcome, Diya. If you have any more questions or need further assistance, feel free to ask. Have a great day!


In [29]:
chat_history

[AIMessage(content='Hello Diya, I wanted to talk about your loan repayment.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='How are you doing today?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 870, 'total_tokens': 876, 'completion_tokens': 6}, 'model': 'mistral-large-latest', 'finish_reason': 'stop'}, id='run-18bfd9c5-69a0-4b30-a8fb-c25f27acd52c-0', usage_metadata={'input_tokens': 870, 'output_tokens': 6, 'total_tokens': 876}),
 HumanMessage(content='What is the time right now?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2QtkFsq3D', 'function': {'name': 'current_date_time', 'arguments': '{}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 886, 'total_tokens': 904, 'completion_tokens': 18}, 'model': 'mistral-large-latest', 'finish_reason': 'tool_calls'}, id='run-dba3dca5-e9c